# SC3000 Direct Preference Optimization — Arithmetic and Algebra

**Selected submission model:** Round9 Random E5, 8.84M-parameter NanoGPT. The instructor's original **Step 1–Step 8** structure, helper functions, loading code, training objective and test questions are retained. The inserted cells complete data preparation, reproduce the selected staged experiment when its frozen data is available, and print actual arithmetic answers.

**Task 1:** preference synthesis and data integrity. **Task 2:** original Course Preference Loss and staged continuation. **Task 3:** print the unchanged instructor test questions. **Task 4:** methods, interpretation, experiment limits and source acknowledgement.

**Run All:** A full historical replay uses the original stage datasets and takes 13 epochs. A standalone course-mode path uses the required `dpo/pos_neg_pairs.json` and executes five standard SFT→DPO epochs if those extra stage datasets are not present. The two are explicitly labeled; a single-stage retrain is **not** described as the historical E5. A short smoke mode is provided to validate the Notebook without replacing the formal checkpoint. This ensures the essential submission files can be executed in order while the full research lineage remains reproducible when its source data is supplied.

### Step 1: Install necesscary packages

In [ ]:
# The instructor's original installation commands, for an unconfigured environment:
# !pip install matplotlib
# !pip install torch numpy transformers datasets tiktoken wandb tqdm
# Use the course/TC1 kernel with its existing CUDA-capable PyTorch installation.
import sys
import torch
print("Kernel:",sys.executable,"Torch:",torch.__version__)

### Step 2: Package imports and configuration

In [ ]:
import sys
import os
from pathlib import Path

# Run this notebook either from the repository root or the dpo/ folder.
# Never depend on a particular user's home-directory location.
work_dir = Path.cwd().resolve()
if (work_dir / "model.py").is_file() and (work_dir / "dpo" / "dpo.ipynb").is_file():
    repo_root = work_dir
elif (work_dir / "dpo.ipynb").is_file() and (work_dir.parent / "model.py").is_file():
    repo_root = work_dir.parent
else:
    raise RuntimeError("Open the notebook from the NanoGPT-Math root or dpo/ directory.")
dpo_dir = repo_root / "dpo"
sys.path.insert(0, str(repo_root))
# Keep Slurm-assigned CUDA_VISIBLE_DEVICES; do not force GPU 1.
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
# Keep 'course' as the historical baseline. Choose 'answer_focused' for Round 1D.
loss_mode = 'course'  # 'course' or 'answer_focused'
answer_weight = 1.0   # chosen numeric-answer negative log-likelihood
completion_weight = 0.25  # chosen full-response NLL to reinforce the output template
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with (repo_root / "sft" / "meta.pkl").open("rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])
print(f"Repository root: {repo_root}")
print(f"Selected device: {device}")


# Reproducible experiment settings (the instructor settings above are unchanged).
FAST_SMOKE = False
TRAINING_POLICY = "auto"      # auto | historical | course
AUTO_SYNTHESIZE_DATA = True
BETA = beta
SEED = 42
BATCH_SIZE = 64
MAX_NEW_TOKENS = 80
TRAIN_BASE_LENGTH = 64
import hashlib, shutil, subprocess, re
from collections import Counter, defaultdict
from datetime import datetime, timezone
RUN_DIR = dpo_dir / "experiments" / (
    "final_notebook_smoke" if FAST_SMOKE else "final_notebook_e5_reproduction")
RUN_DIR.mkdir(parents=True, exist_ok=True)
SFT = repo_root / "sft" / "gpt.pt"
META = repo_root / "sft" / "meta.pkl"
EXPECTED_RANDOM_SHA = "30519fe6883f175e67791a0461afe23e9212f38d974778c4bd5f552d3e26e3fa"
HISTORICAL_ARCHIVE = repo_root/"dpo/experiments/round9_random_multi_e3_e6/epoch_e05.pt"

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as fh:
        for data in iter(lambda:fh.read(1024*1024), b""):
            h.update(data)
    return h.hexdigest()
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))
def save_json_once(path, obj):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite {path}")
    path.write_text(json.dumps(obj,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")

env_record={
    "python":sys.version,"torch":torch.__version__,
    "cuda":torch.version.cuda,
    "device":device,
    "sft_sha256":sha256(SFT),
    "tokenizer_sha256":sha256(META)}
print("Experiment environment:",json.dumps(env_record,indent=2))

### Step 3: Define helper functions

In [3]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

def answer_focused_masks(input_ids):
    """Return masks over next-character targets for the response and number.

    Every preference row has format:
       <math prompt ending with '=?'> The answer is <integer>.
    Question/prompt token probabilities are deliberately excluded.
    Integer answers match the original model.py period-based stopping rule.
    """
    if input_ids.ndim != 2 or input_ids.size(1) < 3:
        raise ValueError("Expected [batch, seq] token IDs")
    if '=' not in stoi or '?' not in stoi:
        raise ValueError("Tokenizer must support the math question marker =?")
    marker = ((input_ids[:, :-1] == stoi['=']) &
              (input_ids[:, 1:] == stoi['?']))
    if not bool(marker.any(dim=1).all().item()):
        raise ValueError("Every preference example must contain =?")
    B, L = input_ids.shape
    # marker[j] means the '?' is character j+1 in the full text.
    absolute_positions = torch.arange(1, L, device=input_ids.device)
    qmark = torch.where(marker, absolute_positions.unsqueeze(0), -1).max(dim=1).values
    valid = input_ids[:, 1:] != 0  # newline ID 0 also serves as padding
    response_mask = (absolute_positions.unsqueeze(0) > qmark[:, None]) & valid
    # The chosen/rejected response is required to start with this shared template.
    template = " The answer is "
    template_ids = torch.tensor(encode(template), dtype=torch.long,
                                device=input_ids.device)
    for i in range(B):
        start = int(qmark[i].item()) + 1
        if start + len(template) > L or not torch.equal(
            input_ids[i, start:start + len(template)], template_ids
        ):
            raise ValueError("Round 1D requires the response template ' The answer is '")
    number_start = qmark + 1 + len(template)
    # Exclude the final period from the numerical NLL: prioritize the sign/digits.
    number_mask = ((absolute_positions.unsqueeze(0) >= number_start[:, None]) &
                   valid & (input_ids[:, 1:] != stoi['.']))
    if not bool(number_mask.any(dim=1).all().item()):
        raise ValueError("Every preference response needs a numeric answer")
    return response_mask, number_mask


def compute_answer_focused_logprob(input_ids):
    """Teacher-forced scores for the response and its numeric answer only.

    Returns (response_logprob_sum, response_logprob_mean, number_logprob_mean).
    The preference term uses SUM, not mean. Positive NLL focuses on numeric
    answer tokens, with a smaller template/completion supervision term.
    """
    logits, _ = gpt(input_ids[:, :-1], full_seq=True)
    targets = input_ids[:, 1:]
    token_logprobs = F.log_softmax(logits, dim=-1).gather(
        -1, targets.unsqueeze(-1)
    ).squeeze(-1)
    response_mask, number_mask = answer_focused_masks(input_ids)
    response_sum = (token_logprobs * response_mask).sum(dim=1)
    response_mean = response_sum / response_mask.sum(dim=1).clamp_min(1)
    number_sum = (token_logprobs * number_mask).sum(dim=1)
    number_mean = number_sum / number_mask.sum(dim=1).clamp_min(1)
    return response_sum, response_mean, number_mean


### Step 4: Load the pretrained NanoGPT model

In [ ]:
ckpt = torch.load(repo_root / "sft" / "gpt.pt", map_location=device, weights_only=True)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()


### Step 5: Load Data (**students are required to complete this part!**)

**Task 1 — Data synthesis and integrity.** The following pure-Python generator implements the arithmetic and equation preference templates used in this project, including exact answer verification, canonical fact partitioning and deterministic sampling. For the selected historical E5, the original Random100k file is used when available; the generator remains executable for a clean standalone submission. No unrelated experiment scripts are embedded or imported.

In [ ]:
# Step 5A — Synthetic preference data, implemented here for reproducibility.
# The answer is independently verified by integer arithmetic. The same
# underlying fact is assigned to a single train/val/test partition.
# For historical E5, the exact archived Random100k source takes precedence.

CATEGORIES = ("add","subtract","multiply","divide",
              "x_plus_a","a_plus_x","x_minus_a","a_minus_x",
              "x_times_a","a_times_x","x_over_a","a_over_x")
TRAIN_QUOTAS = {
    "add":14000,"subtract":10000,"multiply":11000,"divide":9000,
    "x_plus_a":7000,"a_plus_x":7000,"x_minus_a":7000,"a_minus_x":7000,
    "x_times_a":7000,"a_times_x":7000,"x_over_a":7000,"a_over_x":7000}
assert sum(TRAIN_QUOTAS.values())==100000
COURSE_REFUSAL="Sorry, I don't know."

def arithmetic_key(family,a,b):
    if family=="add":
        return ("add",str(min(a,b)),str(max(a,b)),str(a+b))
    return ("mul",str(min(a,b)),str(max(a,b)),str(a*b))

def fact_partition(key, seed=42):
    raw=json.dumps([seed,*key],separators=(",",":"),ensure_ascii=True).encode()
    point=int.from_bytes(hashlib.sha256(raw).digest()[:8],"big")%10000
    return "train" if point<8000 else ("val" if point<9000 else "test")

def generate_fact_templates(a,b):
    # Each item: (category, expression with =?, numeric answer, verified explanation, fact)
    # Fact canonicalization connects e.g. 3*17, 51/3, x*3=51 and 51/x=17.
    sumv=a+b
    addkey=arithmetic_key("add",a,b)
    yield ("add",f"{a}+{b}=?",sumv,f"{a}+{b} equals {sumv}",addkey)
    yield ("subtract",f"{sumv}-{b}=?",a,f"{sumv}-{b} equals {a}",addkey)
    yield ("x_plus_a",f"x+{b}={sumv},x=?",a,f"{sumv}-{b} equals {a}",addkey)
    yield ("a_plus_x",f"{b}+x={sumv},x=?",a,f"{sumv}-{b} equals {a}",addkey)
    yield ("x_minus_a",f"x-{b}={a},x=?",sumv,f"{a}+{b} equals {sumv}",addkey)
    yield ("a_minus_x",f"{sumv}-x={b},x=?",a,f"{sumv}-{b} equals {a}",addkey)
    mult=a*b
    if mult>999:return
    mulkey=arithmetic_key("mul",a,b)
    yield ("multiply",f"{a}*{b}=?",mult,f"{a}*{b} equals {mult}",mulkey)
    yield ("divide",f"{mult}/{b}=?",a,f"{mult}/{b} equals {a}",mulkey)
    yield ("x_times_a",f"x*{b}={mult},x=?",a,f"{mult}/{b} equals {a}",mulkey)
    yield ("a_times_x",f"{b}*x={mult},x=?",a,f"{mult}/{b} equals {a}",mulkey)
    yield ("x_over_a",f"x/{b}={a},x=?",mult,f"{a}*{b} equals {mult}",mulkey)
    yield ("a_over_x",f"{mult}/x={b},x=?",a,f"{mult}/{b} equals {a}",mulkey)

def verify_explanation(explanation,expected):
    m=re.fullmatch(r"(\d+)([+\-*/])(\d+) equals (\d+)",explanation)
    if not m:return False
    a,op,b,r=(int(m.group(1)),m.group(2),int(m.group(3)),int(m.group(4)))
    if op=="+":value=a+b
    elif op=="-":value=a-b
    elif op=="*":value=a*b
    else:value=(a//b if b and a%b==0 else None)
    return value==r==int(expected)

def make_preference_pairs(n=100000,seed=42):
    if n<12:raise ValueError("At least one example per category required")
    rng=random.Random(seed)
    reserved={
        arithmetic_key("add",17,19),
        arithmetic_key("mul",3,17),
        arithmetic_key("mul",4,18),
        arithmetic_key("add",34,38),
        arithmetic_key("mul",4,11)}
    pools={category:{} for category in CATEGORIES}
    for a in range(1,100):
        for b in range(1,100):
            for category,prompt,answer,explanation,key in generate_fact_templates(a,b):
                if key in reserved or fact_partition(key)!="train":
                    continue
                if not verify_explanation(explanation,answer):
                    raise RuntimeError(f"Invalid generated arithmetic: {prompt}")
                pos=f"{prompt} The answer is {answer} because {explanation}."
                neg=f"{prompt} {COURSE_REFUSAL}"
                pools[category][prompt]={"positive":pos,"negative":neg}
    if any(not pool for pool in pools.values()):
        raise RuntimeError("Empty arithmetic template category")
    quotas=TRAIN_QUOTAS if n==100000 else {
        category:n//12+(i<n%12) for i,category in enumerate(CATEGORIES)}
    rows=[]
    for category in CATEGORIES:
        rows.extend(rng.choices(list(pools[category].values()),
                                k=quotas[category]))
    rng.shuffle(rows)
    unique_prompts=len({r["positive"].split(" The answer is ",1)[0] for r in rows})
    print(f"SYNTHESIS: {len(rows)} verified preference pairs, "
          f"{unique_prompts} distinct math prompts; seed={seed}")
    return rows

# Pure-Python generator self-test: check format, math truth and split safety.
_data_smoke=make_preference_pairs(n=24,seed=42)
assert len(_data_smoke)==24
assert all(" The answer is " in x["positive"] and
           x["negative"].endswith("Sorry, I don't know.") for x in _data_smoke)
print("Task 1 code self-test PASS; no external data generator is required.")

**Task 1 — Final training data.** The required `dpo/pos_neg_pairs.json` is checked for 100,000 positive/negative examples. When the frozen Random E5 data exists, its SHA256 is verified and copied to the mandatory submission path (with an original-data backup). Otherwise an existing 100k file is used, or the self-contained generator creates it. The instructor's original Step 5 loading code then reads and validates exactly that file.

In [ ]:
# Step 5B — Choose the final submitted data and protect existing files.
# On TC1, install the frozen Random E5 source as dpo/pos_neg_pairs.json;
# in a submitted three-file package, reuse its 100k preference JSON.
target=dpo_dir/"pos_neg_pairs.json"
frozen=repo_root/"dpo/data/round9_balanced_full/random_pairs.json"
historical_dirs=[
    repo_root/"dpo/data/round3_structured_full/pos_neg_pairs.json",
    repo_root/"dpo/data/round4_matched_full/treatment_pairs.json",
    repo_root/"dpo/data/round5_decomposition/original_pairs.json",
    repo_root/"dpo/data/round7_targeted_full/replay_pairs.json",
    frozen
]
historical_ready=(
    all(p.is_file() for p in historical_dirs) and
    sha256(frozen)==EXPECTED_RANDOM_SHA)
if TRAINING_POLICY not in ("auto","course","historical"):
    raise ValueError("TRAINING_POLICY must be auto, course or historical")
TRAIN_MODE=("historical" if historical_ready else "course"
            ) if TRAINING_POLICY=="auto" else TRAINING_POLICY
if TRAIN_MODE=="historical" and not historical_ready:
    raise RuntimeError("Full E5 replay requires the frozen Round3/R4/R5/R7/R9 data.")

original_file_count=0
if target.exists():
    original_file_count=len(read_json(target))
if historical_ready:
    # dpo/pos_neg_pairs.json is the mandatory ZIP artifact. Keep it identical
    # to the selected E5's real 100k Random training data.
    if not target.exists() or sha256(target)!=EXPECTED_RANDOM_SHA:
        if target.exists():
            backup=RUN_DIR/"original_pos_neg_pairs_before_final.json"
            if not backup.exists():shutil.copy2(target,backup)
            print("Preserved prior submission data:",backup)
        shutil.copy2(frozen,target)
    DATA_ORIGIN="historical_round9_random"
elif target.is_file() and original_file_count==100000:
    DATA_ORIGIN="submission_preference_pairs"
else:
    if not AUTO_SYNTHESIZE_DATA:
        raise FileNotFoundError("Supply dpo/pos_neg_pairs.json with 100k pairs")
    if target.exists():
        backup=RUN_DIR/"original_pos_neg_pairs_before_final.json"
        if not backup.exists():shutil.copy2(target,backup)
    rows=make_preference_pairs(100000,42)
    target.write_text(json.dumps(rows,ensure_ascii=False,indent=2)+"\n",
                      encoding="utf-8")
    DATA_ORIGIN="notebook_generated_from_math_facts"

rows=read_json(target)
if len(rows)!=100000:raise RuntimeError("Expected 100k preference pairs")
if historical_ready and sha256(target)!=EXPECTED_RANDOM_SHA:
    raise RuntimeError("Source and submitted data differ")
DATA_SHA=sha256(target)
if not all(set(x)=={"positive","negative"} for x in rows):
    raise RuntimeError("Preference pair keys changed")
print("Task 1 data ready:",
      {"rows":len(rows),"source":DATA_ORIGIN,"sha256":DATA_SHA,
       "mode":TRAIN_MODE,"positive":rows[0]["positive"],
       "negative":rows[0]["negative"]})
if TRAIN_MODE=="course":
    # Keep an existing final submission checkpoint safe before the course
    # training loop writes its new dpo.pt. This is especially important when
    # the grader re-runs a ZIP that already includes historical E5 weights.
    existing_model=dpo_dir/"dpo.pt"
    if not FAST_SMOKE and existing_model.is_file():
        old_copy=RUN_DIR/"checkpoint_before_course_retrain.pt"
        if not old_copy.is_file():
            shutil.copy2(existing_model,old_copy)
        print("Preserved original submission model:",old_copy)
    print("Course-mode fine-tuning starts from SFT; it is NOT archival E5 reproduction.")


In [ ]:
# Step 5: load and validate preference pairs.
# The supplied pos_neg_pairs.json has only four DEMO examples.
# Replace/extend it with a larger, checked preference dataset for full training.
data_path = dpo_dir / "pos_neg_pairs.json"
if not data_path.exists():
    data_path = dpo_dir / "data" / "pos_neg_pairs.json"
if not data_path.exists():
    raise FileNotFoundError("Missing dpo/pos_neg_pairs.json")

with data_path.open("r", encoding="utf-8") as f:
    raw_pairs = json.load(f)
if not isinstance(raw_pairs, list) or not raw_pairs:
    raise ValueError("Expected a nonempty JSON array of positive/negative pairs")

lines = []
for index, pair in enumerate(raw_pairs):
    if not isinstance(pair, dict) or not all(k in pair for k in ("positive", "negative")):
        raise ValueError(f"Sample {index} must contain positive and negative strings")
    pos, neg = pair["positive"], pair["negative"]
    if not isinstance(pos, str) or not isinstance(neg, str) or pos == neg:
        raise ValueError(f"Invalid responses at sample {index}")

    # The question ends with '=?' in the course's arithmetic/algebra format.
    # This is more reliable than finding the longest common prefix: two numeric
    # answers can share initial digits, which would mask away answer tokens.
    end = pos.find("=?")
    if end < 0:
        raise ValueError(f"Sample {index}: missing '=?' in positive response")
    prompt = pos[:end + 2]
    if not neg.startswith(prompt):
        raise ValueError(f"Sample {index}: positive/negative prompts do not match")
    for response in (pos, neg):
        missing = sorted(set(response) - set(stoi))
        if missing:
            raise ValueError(f"Sample {index} contains unknown characters: {missing}")
    lines.append({"positive": pos, "negative": neg, "prompt": prompt})

max_pair_length = max(max(len(pair["positive"]), len(pair["negative"])) for pair in lines)
# The course get_batches() appends four newline padding markers before packing.
required_length = max_pair_length + 4
if required_length > gpt.config.block_size:
    raise ValueError(
        f"Longest training sequence ({required_length}) exceeds "
        f"model block_size ({gpt.config.block_size})"
    )
max_length = max(max_length, required_length)  # Do not truncate the question.
if len(lines) < batch_size:
    print(f"Only {len(lines)} pairs: shrinking batch_size for a pipeline smoke test.")
    batch_size = len(lines)
print(f"Loaded {len(lines)} preference pairs from {data_path}")
print(f"batch_size={batch_size}, max_length={max_length}")
if len(lines) <= 4:
    print("WARNING: These are the four course examples, NOT a real training dataset.")


**Step 5 data validation.** The original tokenizer, prompt-matching and block-size validations above remain unchanged. Smoke mode truncates only the training *view*, not the submitted 100k JSON.

In [ ]:
# Verify the source used in the instructor's original Step 5 and enable smoke.
assert data_path.resolve()==(dpo_dir/"pos_neg_pairs.json").resolve()
assert len(lines)==100000
assert sha256(data_path)==DATA_SHA
if FAST_SMOKE:
    lines=lines[:128]
    batch_size=16
    epochs=1
    print("FAST_SMOKE: 128 pairs, 1 training epoch; formal checkpoint preserved.")
print("Task 1 audit:",len(lines),"rows loaded by instructor code",
      "full data SHA256",DATA_SHA)

### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [ ]:
# Step 6: build the optimizer and learning-rate scheduler.
# No separate SFT training or reference model is needed for the course loss.
import math

optimizer = torch.optim.AdamW(
    gpt.parameters(),
    lr=base_lr,
    weight_decay=0.01,
)

# The course-provided get_batches() skips the final incomplete batch.
steps_per_epoch = len(lines) // batch_size
if steps_per_epoch == 0:
    raise ValueError("No full batch available: reduce batch_size or add preference data.")
if len(lines) % batch_size:
    print(
        f"Note: {len(lines) % batch_size} example(s) per epoch will be skipped "
        "by the instructor's get_batches() helper."
    )
total_steps = epochs * steps_per_epoch
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=base_lr * 0.1
)
print(f"AdamW lr={base_lr}, epochs={epochs}, steps_per_epoch={steps_per_epoch}")


### Step 7: Begin training (**students are required to complete this part!**)
    
**Two reproducible loss modes:** `course` preserves the instructor's original
whole-sequence mean objective. `answer_focused` uses completion-only
preference sums plus extra correct-number and completion NLL terms. The latter
is a controlled additional experiment and is not standard reference-model DPO.


In [ ]:
# Step 7: Instructor's original Course Loss training loop.
# Execute it in standalone course mode. Historical E5 mode uses the
# same loss with its recorded multi-stage optimizer/scheduler restarts.
RUN_SINGLE_STAGE_DEMO=(TRAIN_MODE=="course")
if RUN_SINGLE_STAGE_DEMO:
    # Step 7: course baseline OR optional Round 1D answer-focused preference loss.
    # The original course branch below is preserved for reproducibility.
    # The new branch is a documented loss experiment, not standard reference-model DPO.
    random.seed(42)
    torch.manual_seed(42)
    gpt.train()
    training_loss = []

    if loss_mode not in ('course', 'answer_focused'):
        raise ValueError("loss_mode must be 'course' or 'answer_focused'")

    for epoch in range(epochs):
        running_loss = 0.0
        completed_batches = 0
        pbar = tqdm(
            get_batches(lines, batch_size),
            total=steps_per_epoch,
            desc=f"Epoch {epoch + 1}/{epochs}",
        )

        for neg_tensor, pos_tensor in pbar:
            optimizer.zero_grad(set_to_none=True)
            if loss_mode == 'course':
                # Instructor's unmodified whole-sequence mean preference loss.
                neg_logprob = compute_logprob(neg_tensor)
                pos_logprob = compute_logprob(pos_tensor)
                preference_loss = -F.logsigmoid(
                    (pos_logprob - neg_logprob) / beta
                ).mean()
                chosen_bonus = -0.1 * pos_logprob.mean()
            else:
                # Only completion tokens contribute to preference comparisons.
                pos_sum, pos_completion_mean, pos_number_mean = (
                    compute_answer_focused_logprob(pos_tensor)
                )
                neg_sum, _, _ = compute_answer_focused_logprob(neg_tensor)
                preference_loss = -F.logsigmoid(
                    (pos_sum - neg_sum) / beta
                ).mean()
                # Chosen token supervision: strong on digits, lighter on template.
                chosen_bonus = (
                    -answer_weight * pos_number_mean.mean()
                    -completion_weight * pos_completion_mean.mean()
                )
            loss = preference_loss + chosen_bonus

            if not torch.isfinite(loss).item():
                raise RuntimeError(f"Non-finite loss at epoch {epoch + 1}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gpt.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            completed_batches += 1
            running_loss += loss.item()
            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                pref=f"{preference_loss.item():.4f}",
                chosen=f"{chosen_bonus.item():.4f}",
            )

        if completed_batches == 0:
            raise RuntimeError("Training did not process any batch")
        mean_loss = running_loss / completed_batches
        training_loss.append(mean_loss)
        print(f"Epoch {epoch + 1}/{epochs}: mean loss={mean_loss:.4f} mode={loss_mode}")

        ckpt_path = (RUN_DIR/"smoke_course.pt" if FAST_SMOKE else dpo_dir/"dpo.pt")
        torch.save({
            "model_state_dict": gpt.state_dict(),
            "model_args": ckpt["model_args"],
        }, ckpt_path)
        print(f"Saved checkpoint to {ckpt_path}")

    print("Training completed.")
    print("Important: loss alone is not arithmetic/algebra accuracy.")

else:
    print("Historical E5 mode: use the multi-stage trainer below.")


**E5 stage continuation.** The instructor's Step 7 training cell above is retained. When frozen historical stage data exists, it runs the original seven-stage Course Loss protocol (13 total epochs) and saves each epoch. In standalone course mode, the original five-epoch cell above trains from the provided SFT checkpoint and required `pos_neg_pairs.json`. These are distinct reproducibility modes; their results are not conflated.

In [ ]:
# Step 3 — Original teacher-forced token log-probabilities and batch helpers
if TRAIN_MODE=="historical":
    from generate_round1 import parse_relation,stable_split
else:
    # The staged dataset verifier is only used in historical mode.
    parse_relation=None
    stable_split=None

def stage_encode(s):
    return [stoi[ch] for ch in s]

def stage_pad_or_truncate(chars, max_length):
    return chars[-max_length:] if len(chars)>max_length else chars+[0]*(max_length-len(chars))

def stage_compute_logprob(gpt, batch_ids):
    inputs = batch_ids[:, :-1]
    targets = batch_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    loss = F.cross_entropy(logits.reshape(-1,logits.size(-1)),
                           targets.reshape(-1), ignore_index=0,
                           reduction="none").reshape_as(targets)
    mask = (targets!=0).float()
    return -(loss*mask).sum(dim=1)/mask.sum(dim=1)

def stage_course_preference_loss(gpt, negatives, positives):
    neg_lp = stage_compute_logprob(gpt, negatives)
    pos_lp = stage_compute_logprob(gpt, positives)
    return -F.logsigmoid((pos_lp-neg_lp)/BETA).mean()-0.1*pos_lp.mean()

def stage_get_batches(lines, batch_size, max_length, target_device):
    # EXACT instructor rule: shuffle rows in place, discard trailing <batch.
    random.shuffle(lines)
    for i in range(0,len(lines),batch_size):
        batch=lines[i:i+batch_size]
        if len(batch)<batch_size:continue
        neg=[stage_pad_or_truncate(stage_encode(x["negative"]+"\n\n\n\n"),max_length) for x in batch]
        pos=[stage_pad_or_truncate(stage_encode(x["positive"]+"\n\n\n\n"),max_length) for x in batch]
        yield (torch.tensor(neg,dtype=torch.long,device=target_device),
               torch.tensor(pos,dtype=torch.long,device=target_device))

def verify_pairs(path, expected_count=100000, smoke=False):
    source = read_json(path)
    if len(source)!=expected_count:
        raise RuntimeError(f"{path}: expected {expected_count}, found {len(source)}")
    rows = source[:128] if smoke else source
    max_required = TRAIN_BASE_LENGTH
    unique_prompts=set()
    unique_facts=set()
    for i,row in enumerate(rows):
        pos,neg=row["positive"],row["negative"]
        marker=" The answer is "
        if marker not in pos or " because " not in pos:
            raise ValueError(f"{path.name} row {i}: incorrect chosen format")
        question,answer_suffix=pos.split(marker,1)
        answer=answer_suffix.split(" because ",1)[0]
        if not re.fullmatch(r"[+-]?\d+",answer) or not pos.endswith("."):
            raise ValueError(f"{path.name} row {i}: malformed integer chosen")
        if neg!=question+" Sorry, I don't know.":
            raise ValueError(f"{path.name} row {i}: changed rejection/prompt")
        if set(pos+neg)-set(stoi):
            raise ValueError(f"{path.name} row {i}: tokenizer OOV")
        max_required=max(max_required,len(pos)+4,len(neg)+4)
        unique_prompts.add(question)
        unique_facts.add(parse_relation(question,answer))
    # Original NanoGPT block size=256; 64 was only the initial pack length.
    if max_required>256:
        raise ValueError(f"{path.name}: {max_required} exceeds model block_size")
    info={"source":str(path.relative_to(repo_root)),"sha256":sha256(path),
          "rows":len(rows),"unique_prompts":len(unique_prompts),
          "unique_facts":len(unique_facts),"max_length":max_required}
    return rows,info

def model_from_checkpoint(path):
    obj=torch.load(path,map_location="cpu",weights_only=True)
    args=obj["model_args"]
    model=GPT(GPTConfig(**args))
    state=obj.get("model_state_dict",obj.get("model"))
    if state is None:raise RuntimeError(f"{path}: missing state dict")
    model.load_state_dict({k.removeprefix("_orig_mod."):v for k,v in state.items()},strict=True)
    return model,args

# Sanity checks: answer-first comparison, no generated completion needed in training.
assert stage_encode("3*17=?")
assert abs(float(F.logsigmoid(torch.tensor(0.0)))+math.log(2))<1e-6
print("PASS: course loss, tokenizer, model loading and batch conventions",flush=True)

In [ ]:
# Step 6 — Historical optimizer and scheduler hyperparameters
R3 = repo_root / "dpo/data/round3_structured_full"
R4 = repo_root / "dpo/data/round4_matched_full"
R5 = repo_root / "dpo/data/round5_decomposition"
R7 = repo_root / "dpo/data/round7_targeted_full"
R9 = repo_root / "dpo/data/round9_balanced_full"

STAGE1_TO_4=[
    {"id":"round3_structured", "data":R3/"pos_neg_pairs.json",
     "manifest":R3/"manifest.json","hash_section":"sha256","hash_key":"train",
     "lr":1e-4,"actual_epochs":5,"schedule_epochs":5},
    {"id":"round4_treatment_e1","data":R4/"treatment_pairs.json",
     "manifest":R4/"manifest.json","hash_section":"file_sha256","hash_key":"treatment",
     "lr":2e-5,"actual_epochs":1,"schedule_epochs":2},
    {"id":"round5_original","data":R5/"original_pairs.json",
     "manifest":R5/"manifest.json","hash_section":"file_sha256","hash_key":"original",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
    {"id":"round7_replay","data":R7/"replay_pairs.json",
     "manifest":R7/"manifest.json","hash_section":"file_sha256","hash_key":"replay",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
]
EXPECTED_RANDOM_SHA="30519fe6883f175e67791a0461afe23e9212f38d974778c4bd5f552d3e26e3fa"



In [ ]:
# Step 6 — Instructor's original AdamW + cosine scheduler, made reusable
# because the chosen E5 was trained in SEVEN sequential stages.
def build_optimizer_scheduler(gpt,base_lr,steps_per_epoch,schedule_epochs):
    optimizer = torch.optim.AdamW(gpt.parameters(),lr=base_lr,weight_decay=0.01)
    total_steps=schedule_epochs*steps_per_epoch
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,T_max=total_steps,eta_min=base_lr*0.1)
    return optimizer,scheduler

print("Step 6 PASS: original AdamW, cosine annealing, fresh restart at each stage.")

In [ ]:
# Step 7 — Course Loss training loop with epoch-boundary resume.
def atomic_save_torch(target, obj):
    target=Path(target)
    if target.exists():
        raise FileExistsError(f"Refusing to overwrite checkpoint: {target}")
    tmp=target.with_suffix(target.suffix+".partial")
    if tmp.exists():
        raise RuntimeError(f"Incomplete temporary checkpoint: {tmp}. Inspect first.")
    torch.save(obj, tmp)
    tmp.rename(target)

def stage_paths(stage_id):
    folder=RUN_DIR/stage_id
    return folder,folder/"progress.json"

def stage_finished_checkpoint(spec):
    folder,_=stage_paths(spec["id"])
    n=1 if FAST_SMOKE else spec["actual_epochs"]
    return folder/f"epoch_{n:02d}.pt"

def train_stage(spec, parent_path):
    assert Path(parent_path).is_file(),f"Missing parent checkpoint {parent_path}"
    data_file=Path(spec["data"])
    source_m=read_json(spec["manifest"])
    expected=source_m[spec["hash_section"]][spec["hash_key"]]
    if sha256(data_file)!=expected:
        raise RuntimeError(f"Data SHA256 does not match frozen source manifest: {data_file}")
    allrows,info=verify_pairs(data_file,smoke=FAST_SMOKE)
    batch=16 if FAST_SMOKE else BATCH_SIZE
    actual=1 if FAST_SMOKE else spec["actual_epochs"]
    schedule_epochs=spec["schedule_epochs"]
    steps=len(allrows)//batch
    if steps<=0:raise RuntimeError("Not enough preferences for one minibatch")
    protocol={
        "stage":spec["id"],"parent_sha256":sha256(parent_path),
        "dataset_sha256":info["sha256"],
        "source_manifest_sha256":sha256(spec["manifest"]),
        "seed":SEED,"batch_size":batch,"steps_per_epoch":steps,
        "actual_epochs":actual,"cosine_schedule_epochs":schedule_epochs,
        "learning_rate":spec["lr"],"loss":"instructor full-sequence mean",
        "max_length":info["max_length"],"beta":BETA,
        "model_block_size":256,"fast_smoke":FAST_SMOKE
    }
    stage_dir,progress_path=stage_paths(spec["id"])
    if progress_path.is_file():
        progress=read_json(progress_path)
        if progress["protocol"]!=protocol:
            raise RuntimeError("Refusing incompatible resume: stage data/schedule/parent changed")
        finished=progress["completed_epochs"]
        if not 1<=finished<=actual:
            raise RuntimeError("Invalid saved epoch offset")
        epoch_file=stage_dir/f"epoch_{finished:02d}.pt"
        if sha256(epoch_file)!=progress["last_checkpoint_sha256"]:
            raise RuntimeError(f"Corrupt stage checkpoint {epoch_file}")
        if finished==actual:
            print(f"SKIP verified {spec['id']}: {finished}/{actual} complete",flush=True)
            return epoch_file
        print(f"RESUME {spec['id']} from epoch {finished}/{actual}",flush=True)
    elif stage_dir.exists():
        # A failed allocation before the first optimizer update may have
        # written protocol.json but no checkpoint yet. Restart that *empty*
        # stage safely without overwriting any saved model.
        existing={p.name for p in stage_dir.iterdir()}
        if existing=={"protocol.json"} and read_json(stage_dir/"protocol.json")==protocol:
            finished=0
            progress=None
            print(f"RESTART EMPTY STAGE {spec['id']}: no completed epoch lost",flush=True)
        else:
            raise RuntimeError(f"Stale/incomplete stage folder: {stage_dir}; "
                               "inspect or rename; never overwrite saved epochs")
    else:
        stage_dir.mkdir(parents=True,exist_ok=False)
        save_json_once(stage_dir/"protocol.json",protocol)
        finished=0
        progress=None

    # Previous Python/CUDA seed reset BEFORE constructing GPT, just like all
    # historical train_round*.py stage entry points.
    random.seed(SEED)
    torch.manual_seed(SEED)
    if device=="cuda":torch.cuda.manual_seed_all(SEED)
    gpt, model_args=model_from_checkpoint(parent_path)
    if model_args["vocab_size"]!=len(stoi) or model_args["block_size"]!=256:
        raise RuntimeError("Original character tokenizer / model architecture changed")
    gpt.to(device).train()
    opt,scheduler=build_optimizer_scheduler(gpt,spec["lr"],steps,schedule_epochs)
    lines=allrows

    if finished:
        ck=torch.load(epoch_file,map_location="cpu",weights_only=True)
        if ck["stage_protocol"]!=protocol or ck["epoch"]!=finished:
            raise RuntimeError("Resume checkpoint metadata mismatch")
        gpt.load_state_dict(ck["model_state_dict"],strict=True)
        opt.load_state_dict(ck["optimizer_state_dict"])
        scheduler.load_state_dict(ck["scheduler_state_dict"])
        lines=ck["shuffled_lines"]
        random.setstate(ck["python_rng_state"])
        torch.set_rng_state(ck["torch_rng_state"])
        if device=="cuda":
            torch.cuda.set_rng_state_all(ck["cuda_rng_states"])
        if len(lines)!=len(allrows):
            raise RuntimeError("Saved shuffle order lost rows")

    for epoch in range(finished+1,actual+1):
        gpt.train()
        accumulated=0.
        updated=0
        for step,(neg,pos) in enumerate(stage_get_batches(lines,batch,info["max_length"],device),1):
            opt.zero_grad(set_to_none=True)
            loss=stage_course_preference_loss(gpt,neg,pos)
            if not bool(torch.isfinite(loss).item()):
                raise RuntimeError(f"Non-finite loss: {spec['id']} epoch={epoch} step={step}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gpt.parameters(),max_norm=1.0)
            opt.step()
            scheduler.step()
            updated+=1
            accumulated+=float(loss.item())
            if step==1 or step%250==0 or step==steps:
                print(f"TRAIN {spec['id']} epoch={epoch}/{actual} "
                      f"step={step}/{steps} loss={loss.item():.5f} "
                      f"lr={scheduler.get_last_lr()[0]:.9g}",flush=True)
        if updated!=steps:raise RuntimeError("Optimizer step count mismatch")
        ckpt=stage_dir/f"epoch_{epoch:02d}.pt"
        obj={
            "epoch":epoch,"stage_protocol":protocol,
            "model_state_dict":gpt.state_dict(),"model_args":model_args,
            "optimizer_state_dict":opt.state_dict(),
            "scheduler_state_dict":scheduler.state_dict(),
            "shuffled_lines":lines,"python_rng_state":random.getstate(),
            "torch_rng_state":torch.get_rng_state(),
            "cuda_rng_states":torch.cuda.get_rng_state_all() if device=="cuda" else []
        }
        atomic_save_torch(ckpt,obj)
        history=progress["history"] if progress else []
        history.append({"epoch":epoch,"mean_loss":accumulated/steps,
                        "checkpoint":ckpt.name,"sha256":sha256(ckpt)})
        progress={"protocol":protocol,"completed_epochs":epoch,
                  "last_checkpoint_sha256":sha256(ckpt),"history":history}
        temp=progress_path.with_suffix(".json.partial")
        if temp.exists():raise RuntimeError(f"Stale progress marker: {temp}")
        temp.write_text(json.dumps(progress,indent=2)+"\n",encoding="utf-8")
        temp.replace(progress_path)
        print(f"EPOCH SAVED {spec['id']} {epoch}/{actual} "
              f"mean_loss={accumulated/steps:.6f}",flush=True)
    print(f"STAGE PASS {spec['id']} -> {ckpt.name}",flush=True)
    return ckpt

# Verify the original SFT config without changing any model weights.
print("Step 7 training engine PASS: loss, optimizer, checkpoint and RNG resume ready",flush=True)

In [ ]:
if TRAIN_MODE=="historical":
    # Step 6's demonstration optimizer is not needed in the staged replay.
    del optimizer,scheduler,gpt
    if device=="cuda":torch.cuda.empty_cache()
    # Step 7A — Train/reuse Round3, Round4 E1, Round5 Original, Round7 Replay
    stage_outputs={"sft":SFT}
    parent=SFT
    for spec in STAGE1_TO_4:
        parent=train_stage(spec,parent)
        stage_outputs[spec["id"]]=parent
        print(f"CHAIN {spec['id']}: SHA256 {sha256(parent)[:20]}",flush=True)

    ROUND7_PARENT=stage_outputs["round7_replay"]
    print(f"PASS: Round7 Replay predecessor created/verified: {ROUND7_PARENT}",flush=True)
else:
    print("Standalone course mode: the original Step 7 training is already complete.")


In [ ]:
if TRAIN_MODE=="historical":
    # Step 7B — Verify/build the Round9 Random 100k training dataset
    EXISTING_R9_DATA=R9/"random_pairs.json"
    if EXISTING_R9_DATA.exists():
        r9_manifest=read_json(R9/"manifest.json")
        if r9_manifest["version"]!="round9_structured_v1":
            raise RuntimeError("Wrong historical Round9 manifest")
        if sha256(EXISTING_R9_DATA)!=r9_manifest["files_sha256"]["random"]:
            raise RuntimeError("Historical Random100k data SHA256 mismatch")
        random_data_path=EXISTING_R9_DATA
        random_manifest_path=R9/"manifest.json"
    else:
        if not AUTO_BUILD_MISSING_DATA:
            raise FileNotFoundError("Missing Round9 Random preference dataset")
        import build_round9_data as b9
        import evaluate_teacher_style_680 as teacher680_module
        # Strictly validate both existing frozen teacher benchmarks.
        teacher680_module.validate()
        generated=RUN_DIR/"notebook_generated_round9"
        if generated.exists():
            if not (generated/"random_pairs.json").exists():
                raise RuntimeError("Incomplete generated Round9 folder; inspect it")
            random_data_path=generated/"random_pairs.json"
            random_manifest_path=generated/"manifest.json"
        else:
            # The original builder expects the parent at a historical absolute
            # path. Use our new Round7 checkpoint, without copying it there.
            historical_base=b9.BASE
            try:
                b9.BASE=ROUND7_PARENT
                r9_pairs,r9_metadata,r9_manifest=b9.make()
            finally:
                b9.BASE=historical_base
            generated.mkdir(parents=True,exist_ok=False)
            for arm in ("replay","random","structured"):
                (generated/f"{arm}_pairs.json").write_bytes(b9.bytes_json(r9_pairs[arm]))
            (generated/"structured_metadata.json").write_bytes(
                b9.bytes_json(r9_metadata))
            (generated/"manifest.json").write_bytes(b9.bytes_json(r9_manifest))
            random_data_path=generated/"random_pairs.json"
            random_manifest_path=generated/"manifest.json"

    r9_manifest=read_json(random_manifest_path)
    if sha256(random_data_path)!=r9_manifest["files_sha256"]["random"]:
        raise RuntimeError("Generated Random source mismatch with its own manifest")
    if sha256(random_data_path)!=EXPECTED_RANDOM_SHA:
        raise RuntimeError(
            "Round9 Random data do NOT match archived E5 training source. "
            "Do not publish an alleged historical reproduction with changed data. "
            f"observed={sha256(random_data_path)} expected={EXPECTED_RANDOM_SHA}")
    print(f"PASS: historical Random100k exact pair SHA={sha256(random_data_path)}")
    print(f"Dataset location: {random_data_path}",flush=True)
else:
    random_data_path=dpo_dir/"pos_neg_pairs.json"
    print("Standalone training data SHA256:",sha256(random_data_path))


In [ ]:
if TRAIN_MODE=="historical":
    # Step 7C — Continue original staged training until selected E5
    STAGE5_TO_7=[
        {"id":"round9_random_e1", "data":random_data_path,
         "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
         "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
        {"id":"round9_random_e2_restart", "data":random_data_path,
         "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
         "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
        {"id":"round9_random_e3_e5_from_e2", "data":random_data_path,
         "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
         "lr":1e-5,"actual_epochs":3,"schedule_epochs":4}
    ]
    assert [s["actual_epochs"] for s in STAGE5_TO_7]==[1,1,3]
    assert [s["schedule_epochs"] for s in STAGE5_TO_7]==[1,1,4]

    parent=ROUND7_PARENT
    for spec in STAGE5_TO_7:
        parent=train_stage(spec,parent)
        stage_outputs[spec["id"]]=parent
        print(f"CHAIN {spec['id']} SHA256={sha256(parent)[:20]}",flush=True)

    REPRODUCED_E5=stage_outputs["round9_random_e3_e5_from_e2"]
    print(f"FINAL E5 REPLAY CHECKPOINT = {REPRODUCED_E5}",flush=True)
else:
    REPRODUCED_E5=(RUN_DIR/"smoke_course.pt") if FAST_SMOKE else dpo_dir/"dpo.pt"
    print("Course-mode checkpoint:",REPRODUCED_E5)


### Step 8: Begin testing (**students are required to complete this part!**)

#### Step 8A — Select final model and produce test results

The original instructor Step 8 heading is retained above. The training branch chooses the completed fine-tuned checkpoint and measures real development results before printing the unchanged instructor `test_set`.

In [ ]:
# Step 8A — Select the checkpoint generated by the completed training path.
def load_dpo_checkpoint(path):
    ck=torch.load(path,map_location="cpu",weights_only=True)
    model=GPT(GPTConfig(**ck["model_args"]))
    state=ck.get("model_state_dict",ck.get("model"))
    if state is None:raise RuntimeError(f"Missing weights: {path}")
    model.load_state_dict({k.removeprefix("_orig_mod."):v
                           for k,v in state.items()},strict=True)
    return model,ck["model_args"]

parity={"archive_exists":HISTORICAL_ARCHIVE.is_file()}
if TRAIN_MODE=="historical":
    selected=HISTORICAL_ARCHIVE if HISTORICAL_ARCHIVE.is_file() else REPRODUCED_E5
    if HISTORICAL_ARCHIVE.is_file():
        left,_=load_dpo_checkpoint(REPRODUCED_E5)
        right,_=load_dpo_checkpoint(HISTORICAL_ARCHIVE)
        a,b=left.state_dict(),right.state_dict()
        parity["weight_equal"]=all(torch.equal(a[k],b[k]) for k in a)
        parity["max_weight_difference"]=max(
            (a[k]-b[k]).abs().max().item() for k in a)
        del left,right
    if FAST_SMOKE:
        FINAL_CHECKPOINT=REPRODUCED_E5
    else:
        FINAL_CHECKPOINT=dpo_dir/"dpo.pt"
        model,args=load_dpo_checkpoint(selected)
        state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        del model
        overwrite=True
        if FINAL_CHECKPOINT.exists():
            previous,_=load_dpo_checkpoint(FINAL_CHECKPOINT)
            overwrite=any(not torch.equal(previous.state_dict()[k],v)
                          for k,v in state.items())
            del previous
        if overwrite:
            if FINAL_CHECKPOINT.exists():
                backup=RUN_DIR/"previous_dpo_before_e5.pt"
                if not backup.exists():shutil.copy2(FINAL_CHECKPOINT,backup)
            temporary=FINAL_CHECKPOINT.with_suffix(".pt.partial")
            if temporary.exists():raise RuntimeError(f"Partial file: {temporary}")
            torch.save({"model_args":args,"model_state_dict":state},temporary)
            temporary.replace(FINAL_CHECKPOINT)
else:
    FINAL_CHECKPOINT=REPRODUCED_E5
if not Path(FINAL_CHECKPOINT).exists():
    raise RuntimeError("Step 7 did not produce model weights")
print("Selected checkpoint:",FINAL_CHECKPOINT,sha256(FINAL_CHECKPOINT))
print("Historical checkpoint parity:",parity)

In [ ]:
# Step 8B — Greedy next-character inference and answer-first grading.
ANSWER_PATTERN=re.compile(r"^\s*The answer is\s+([+-]?\d+)(?=\s|[.]|$)",re.I)
def greedy_original_stop(model,prompt,limit=80):
    tokens=encode(prompt)
    generated=[]
    stop="max_new_tokens"
    with torch.inference_mode():
        for _ in range(limit):
            x=torch.tensor([tokens[-model.config.block_size:]],
                           dtype=torch.long,device=device)
            logits,_=model(x)
            nxt=int(logits[0,-1].argmax().item())
            if nxt==0:
                stop="newline";break
            tokens.append(nxt);generated.append(nxt)
            if nxt==7:
                stop="period";break
    return "".join(itos[k] for k in generated),stop

def measure(model,records,name):
    results=[];correct=0
    for i,item in enumerate(records):
        completion,stop=greedy_original_stop(model,item["prompt"],MAX_NEW_TOKENS)
        match=ANSWER_PATTERN.match(completion)
        prediction=int(match.group(1)) if match else None
        status=("format_failure" if prediction is None else
                "correct" if prediction==int(item["answer"]) else "math_error")
        correct+=status=="correct"
        results.append({"prompt":item["prompt"],"answer":str(item["answer"]),
                        "completion":completion,"status":status,"stop":stop})
        if i<8:print(item["prompt"]+completion, "["+status+"]")
    summary={"n":len(records),"correct":correct,"accuracy":correct/len(records)}
    print(name,correct,"/",len(records),f"({summary['accuracy']:.2%})")
    return results,summary

instructor_cases=[
    {"prompt":"17+19=?","answer":"36"},
    {"prompt":"3*17=?","answer":"51"},
    {"prompt":"72/4=?","answer":"18"},
    {"prompt":"72-x=34,x=?","answer":"38"},
    {"prompt":"x*11=44,x=?","answer":"4"}]
dev_sets={"instructor_examples":instructor_cases}
r3val=repo_root/"dpo/data/round3_structured_full/val.json"
r3manifest=repo_root/"dpo/data/round3_structured_full/manifest.json"
if r3val.is_file() and r3manifest.is_file():
    assert sha256(r3val)==read_json(r3manifest)["sha256"]["val"]
    dev_sets["val768"]=read_json(r3val)
teacher_dirs=[repo_root/"dpo/benchmarks/teacher_style_400_v1",
              repo_root/"dpo/benchmarks/teacher_style_extension_280_v1"]
if all((p/"questions.json").exists() and (p/"manifest.json").exists()
       for p in teacher_dirs):
    records=[]
    for directory in teacher_dirs:
        assert sha256(directory/"questions.json")==read_json(
            directory/"manifest.json")["questions_sha256"]
        records+=read_json(directory/"questions.json")
    if len(records)==680:dev_sets["teacher680_seen"]=records
if FAST_SMOKE:dev_sets={k:v[:8] for k,v in dev_sets.items()}
measured={}
model,_=load_dpo_checkpoint(FINAL_CHECKPOINT)
model.to(device).eval()
for name,records in dev_sets.items():
    output=RUN_DIR/f"eval_{name}.json"
    summarypath=RUN_DIR/f"eval_{name}_summary.json"
    if output.exists() and summarypath.exists():
        preds=read_json(output);score=read_json(summarypath)
        if len(preds)!=len(records) or score["n"]!=len(records):
            raise RuntimeError("Cached evaluation does not match fixed questions")
    elif output.exists() or summarypath.exists():
        raise RuntimeError("Partial evaluation: "+name)
    else:
        preds,score=measure(model,records,name)
        save_json_once(output,preds)
        save_json_once(summarypath,score)
    measured[name]=score
del model
if device=="cuda":torch.cuda.empty_cache()
print("Only development and instructor examples evaluated, never private Test.")

In [ ]:
# Load the fine-tuned model
ckpt_path = FINAL_CHECKPOINT
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=True)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        # Inference completed using the original greedy character stopping rule.
        completion,stop_reason=greedy_original_stop(gpt,prompt,MAX_NEW_TOKENS)
        print(prompt+completion)
        ###########################################################


#### Step 8B — Experimental analysis and submission notes

**Data construction (Task 1).** For each of the 12 arithmetic/equation classes we use verified Chosen answers and refusal Rejected answers. Related operations are grouped by canonical math facts before seeded data partitioning. The course's required `dpo/pos_neg_pairs.json` is the source actually loaded by Step 5, and its SHA256 is recorded.

**Fine-tuning (Task 2).** The submitted model uses character-level Teacher Forcing and the instructor's original whole-sequence mean preference objective, including the Chosen likelihood bonus. The training method uses AdamW and cosine scheduling. The full historical Random E5 route contains separate curriculum stages and optimizer resets, whereas the standalone course mode is a five-epoch SFT fine-tune and is explicitly labelled as a different experiment.

**Testing (Task 3).** The original instructor's eight test prompts are preserved and printed in the code cell immediately above. The development evaluations are computed from actual outputs and stored beside the Notebook run.

**Results and limitations (Task 4).** In previous frozen development experiments, the selected historical Random E5 obtained 614/768 on Val768 (79.95%) and 521/680 on Teacher680 (76.62%). Teacher680 includes historically seen facts, so it is a retention diagnostic, not proof of unseen-fact generalization. The small model may fail some unfamiliar arithmetic combinations; increased repeated exposure does not guarantee an algorithmic arithmetic procedure. The TA private test has not been used.

**Sources and responsibilities.** The original GPT model, notebook scaffold and course preference objective are from the supplied lab materials. Mathematical data synthesis, stage management, validity checks and additional evaluation are project work. **Fill in the actual contributor(s) and their specific responsibilities before submitting:** [Contributor(s) and contribution(s)].

In [ ]:
# Save actual course outputs and the source of the selected checkpoint.
stage_history={}
if TRAIN_MODE=="historical":
    stage_history={k:{"checkpoint":str(v),"sha256":sha256(v)}
                   for k,v in stage_outputs.items()}
final_report={
    "mode":TRAIN_MODE,"smoke":FAST_SMOKE,
    "training_pairs":100000,"training_data_origin":DATA_ORIGIN,
    "training_data_sha256":DATA_SHA,
    "checkpoint":str(FINAL_CHECKPOINT),
    "checkpoint_sha256":sha256(FINAL_CHECKPOINT),
    "environment":env_record,
    "historical_checkpoint_parity":parity,
    "development_evaluation":measured,
    "stage_lineage":stage_history,
    "ta_private_test_used":False}
report_path=RUN_DIR/"final_report.json"
if report_path.exists():
    earlier=read_json(report_path)
    if (earlier["training_data_sha256"]!=DATA_SHA or
        earlier["checkpoint_sha256"]!=sha256(FINAL_CHECKPOINT)):
        raise RuntimeError("Saved report belongs to a different model or dataset")
else:
    save_json_once(report_path,final_report)
print("Saved reproducibility report:",report_path)